In [ ]:
# 셀 1 — 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:

# 셀 2 — 처음 한 번만 실행 (다운로드 + 드라이브 저장)
import os

DRIVE_PATH = "/content/drive/MyDrive/csic2010"
os.makedirs(DRIVE_PATH, exist_ok=True)

# 드라이브에 없을 때만 다운로드
files = {
    "normalTrafficTraining.txt": f"{DRIVE_PATH}/normalTrafficTraining.txt",
    "anomalousTrafficTest.txt":  f"{DRIVE_PATH}/anomalousTrafficTest.txt",
    "normalTrafficTest.txt":     f"{DRIVE_PATH}/normalTrafficTest.txt",
}

BASE_URL = "https://raw.githubusercontent.com/msudol/Web-Application-Attack-Datasets/master/OriginalDataSets/csic_2010"

for filename, filepath in files.items():
    if not os.path.exists(filepath):
        print(f"다운로드 중: {filename}")
        os.system(f'wget -q "{BASE_URL}/{filename}" -O "{filepath}"')
    else:
        print(f"이미 있음: {filename}")

print("완료!")

이미 있음: normalTrafficTraining.txt
이미 있음: anomalousTrafficTest.txt
이미 있음: normalTrafficTest.txt
완료!


In [ ]:
# 셀 3 — 다음부턴 이것만 실행
DRIVE_PATH = "/content/drive/MyDrive/csic2010"

normal_train_path = f"{DRIVE_PATH}/normalTrafficTraining.txt"
anomalous_path    = f"{DRIVE_PATH}/anomalousTrafficTest.txt"
normal_test_path  = f"{DRIVE_PATH}/normalTrafficTest.txt"

print("드라이브에서 로드 준비 완료!")

드라이브에서 로드 준비 완료!


In [ ]:
# 처음 50줄 확인
with open("normalTrafficTraining.txt", "r", encoding="latin-1") as f:
    for i, line in enumerate(f):
        print(line, end="")
        if i > 50:
            break

GET http://localhost:8080/tienda1/index.jsp HTTP/1.1
User-Agent: Mozilla/5.0 (compatible; Konqueror/3.5; Linux) KHTML/3.5.8 (like Gecko)
Pragma: no-cache
Cache-control: no-cache
Accept: text/xml,application/xml,application/xhtml+xml,text/html;q=0.9,text/plain;q=0.8,image/png,*/*;q=0.5
Accept-Encoding: x-gzip, x-deflate, gzip, deflate
Accept-Charset: utf-8, utf-8;q=0.5, *;q=0.5
Accept-Language: en
Host: localhost:8080
Cookie: JSESSIONID=1F767F17239C9B670A39E9B10C3825F4
Connection: close


GET http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito HTTP/1.1
User-Agent: Mozilla/5.0 (compatible; Konqueror/3.5; Linux) KHTML/3.5.8 (like Gecko)
Pragma: no-cache
Cache-control: no-cache
Accept: text/xml,application/xml,application/xhtml+xml,text/html;q=0.9,text/plain;q=0.8,image/png,*/*;q=0.5
Accept-Encoding: x-gzip, x-deflate, gzip, deflate
Accept-Charset: utf-8, utf-8;q=0.5, *;q=0.5
Accept-Language: en
Host: localhost:8080
Cookie: 

In [ ]:
def parse_single_request(lines):
    if not lines:
        return None
    req = {
        "method": None, "url": None, "path": None,
        "params": {}, "headers": {}, "body": None,
        "user_agent": None, "cookie": None,
        "has_cookie": False,
    }
    try:
        parts = lines[0].strip().split(" ")
        req["method"] = parts[0]
        req["url"]    = parts[1] if len(parts) > 1 else ""

        if "?" in req["url"]:
            req["path"] = req["url"].split("?")[0]
            for param in req["url"].split("?")[1].split("&"):
                if "=" in param:
                    k, v = param.split("=", 1)
                    req["params"][k] = v
        else:
            req["path"] = req["url"]

        body_start = None
        for i, line in enumerate(lines[1:], 1):
            # 빈 문자열이면 다음으로 넘어감
            if line.strip() == "":
                body_start = i + 1
                break
            if ": " in line:
                k, v = line.split(": ", 1)
                k = k.strip().lower()
                req["headers"][k] = v.strip()
                if k == "user-agent":
                    req["user_agent"] = v.strip()
                elif k == "cookie":
                    req["cookie"] = v.strip()
                    req["has_cookie"] = True

        if body_start and body_start < len(lines):
            req["body"] = lines[body_start].strip()
            for param in req["body"].split("&"):
                if "=" in param:
                    k, v = param.split("=", 1)
                    req["params"][k] = v
    except:
        return None
    return req

def parse_requests(filepath):
    requests, current = [], []
    with open(filepath, "r", encoding="latin-1") as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == "":
                if current:
                    req = parse_single_request(current)
                    if req: requests.append(req)
                    current = []
            else:
                current.append(line)
    if current:
        req = parse_single_request(current)
        if req: requests.append(req)
    return requests

print("파싱 함수 정의 완료")

파싱 함수 정의 완료


In [ ]:
normal_requests = parse_requests("normalTrafficTraining.txt")
attack_requests = parse_requests("anomalousTrafficTest.txt")

print(f"정상 요청 수: {len(normal_requests)}")
print(f"공격 요청 수: {len(attack_requests)}")

정상 요청 수: 44000
공격 요청 수: 35042


In [ ]:
# 셀 4 — path 분포 확인
from collections import Counter

paths = [r["path"] for r in normal_requests if r["path"]]
print("=== Path 분포 (상위 20개) ===")
for path, cnt in Counter(paths).most_common(20):
    print(f"  {cnt:5d}  {path}")

=== Path 분포 (상위 20개) ===
   2000  http://localhost:8080/tienda1/publico/anadir.jsp
   2000  http://localhost:8080/tienda1/publico/autenticar.jsp
   2000  http://localhost:8080/tienda1/publico/caracteristicas.jsp
   2000  http://localhost:8080/tienda1/publico/entrar.jsp
   2000  http://localhost:8080/tienda1/publico/pagar.jsp
   2000  http://localhost:8080/tienda1/publico/registro.jsp
   2000  http://localhost:8080/tienda1/publico/vaciar.jsp
   2000  http://localhost:8080/tienda1/miembros/editar.jsp
   1000  http://localhost:8080/tienda1/index.jsp
   1000  http://localhost:8080/tienda1/publico/carrito.jsp
   1000  http://localhost:8080/tienda1/publico/miembros.jsp
   1000  http://localhost:8080/tienda1/publico/productos.jsp
   1000  http://localhost:8080/tienda1/global/creditos.jsp
   1000  http://localhost:8080/tienda1/global/menum.jsp
   1000  http://localhost:8080/tienda1/global/titulo.jsp
   1000  http://localhost:8080/tienda1/global/estilos.css
   1000  http://localhost:8080/tienda

In [ ]:
# 셀 5 — 전이 패턴 확인
transitions = []
for i in range(len(normal_requests) - 1):
    a = normal_requests[i]["path"]
    b = normal_requests[i+1]["path"]
    if a and b:
        transitions.append(f"{a} → {b}")

print("=== 전이 패턴 (상위 20개) ===")
for trans, cnt in Counter(transitions).most_common(20):
    print(f"  {cnt:5d}  {trans}")

=== 전이 패턴 (상위 20개) ===
   1000  http://localhost:8080/tienda1/index.jsp → http://localhost:8080/tienda1/publico/anadir.jsp
   1000  http://localhost:8080/tienda1/publico/anadir.jsp → http://localhost:8080/tienda1/publico/anadir.jsp
   1000  http://localhost:8080/tienda1/publico/autenticar.jsp → http://localhost:8080/tienda1/publico/autenticar.jsp
   1000  http://localhost:8080/tienda1/publico/caracteristicas.jsp → http://localhost:8080/tienda1/publico/caracteristicas.jsp
   1000  http://localhost:8080/tienda1/publico/carrito.jsp → http://localhost:8080/tienda1/publico/entrar.jsp
   1000  http://localhost:8080/tienda1/publico/entrar.jsp → http://localhost:8080/tienda1/publico/entrar.jsp
   1000  http://localhost:8080/tienda1/publico/miembros.jsp → http://localhost:8080/tienda1/publico/pagar.jsp
   1000  http://localhost:8080/tienda1/publico/pagar.jsp → http://localhost:8080/tienda1/publico/pagar.jsp
   1000  http://localhost:8080/tienda1/publico/productos.jsp → http://localhost:8080/tie